# 08 — Segmentación RFM + Diagnóstico de negocio

Dos herramientas del analista que van más allá de mostrar números: la primera clasifica clientes por comportamiento para que el negocio actúe diferente con cada grupo; la segunda estructura el diagnóstico para identificar no solo qué pasó sino por qué y dónde actuar.

## Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT  = find_project_root()
df    = pd.read_csv(ROOT / 'data' / 'external' / 'train.csv', low_memory=False)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%d/%m/%Y')
print(f'Shape: {df.shape}')
print(df[['Customer ID', 'Order Date', 'Sales']].head())


Shape: (9800, 18)
  Customer ID Order Date     Sales
0    CG-12520 2017-11-08  261.9600
1    CG-12520 2017-11-08  731.9400
2    DV-13045 2017-06-12   14.6200
3    SO-20335 2016-10-11  957.5775
4    SO-20335 2016-10-11   22.3680


---
# Bloque 1 — Segmentación RFM

RFM es un framework de segmentación basado en comportamiento de compra, no en demografía. La premisa es que el mejor predictor del comportamiento futuro de un cliente es su comportamiento pasado.

| Dimensión | Pregunta | Cómo se calcula |
|-----------|----------|-----------------|
| **R** ecency | ¿Cuándo compró por última vez? | Días desde la última orden hasta hoy |
| **F** requency | ¿Con qué frecuencia compra? | Número de órdenes únicas |
| **M** onetary | ¿Cuánto ha gastado en total? | Suma de Sales |

Un cliente con R alto, F alto y M alto es el mejor cliente. Un cliente con R bajo (no compra hace mucho), F bajo y M bajo está en riesgo de perderse.

## Calcular R, F y M

In [2]:
# La fecha de referencia es el día siguiente al último pedido del dataset
# Se usa esta fecha en vez de 'hoy' para que el análisis sea reproducible
FECHA_REF = df['Order Date'].max() + pd.Timedelta(days=1)
print(f'Fecha de referencia: {FECHA_REF.date()}')

rfm = (
    df.groupby('Customer ID')
    .agg(
        recency   = ('Order Date', lambda x: (FECHA_REF - x.max()).days),
        frequency = ('Order ID',   'nunique'),
        monetary  = ('Sales',      'sum'),
        nombre    = ('Customer Name', 'first'),
    )
    .reset_index()
    .round({'monetary': 2})
)

print(f'Clientes únicos: {len(rfm)}')
print(rfm.describe().round(1))


Fecha de referencia: 2018-12-31
Clientes únicos: 793
       recency  frequency  monetary
count    793.0      793.0     793.0
mean     149.3        6.2    2851.9
std      187.1        2.5    2620.7
min        1.0        1.0       4.8
25%       31.0        4.0    1081.5
50%       76.0        6.0    2215.0
75%      185.0        8.0    3670.3
max     1166.0       17.0   25043.0


## Puntuar cada dimensión

Se divide cada dimensión en 3 grupos iguales usando `pd.qcut()` (cuantiles). El resultado es un score del 1 al 3 para R, F y M.

El score de **Recency se invierte**: un cliente que compró hace 5 días (R bajo) debe tener score 3 (el mejor), no 1. Alguien que compró hace 300 días tiene score 1 (en riesgo).

In [3]:
# qcut divide en grupos con el mismo número de elementos
# labels=[1,2,3] asigna un entero a cada grupo
# duplicates='drop' maneja casos donde los límites de cuantil son idénticos

rfm['R'] = pd.qcut(rfm['recency'],   q=3, labels=[3, 2, 1], duplicates='drop').astype(int)
rfm['F'] = pd.qcut(rfm['frequency'],  q=3, labels=[1, 2, 3], duplicates='drop').astype(int)
rfm['M'] = pd.qcut(rfm['monetary'],   q=3, labels=[1, 2, 3], duplicates='drop').astype(int)

# Score compuesto: suma de R+F+M → rango 3 (peor) a 9 (mejor)
rfm['rfm_score'] = rfm['R'] + rfm['F'] + rfm['M']

print('Distribución de scores:')
print(rfm['rfm_score'].value_counts().sort_index())


Distribución de scores:
rfm_score
3    101
4    112
5    138
6    130
7    136
8    111
9     65
Name: count, dtype: int64


## Clasificar en segmentos

In [4]:
def segmentar(row):
    r, f, m = row['R'], row['F'], row['M']

    if r == 3 and f == 3 and m == 3:
        return 'Campeones'           # reciente, frecuente, alto gasto
    elif r == 3 and f >= 2:
        return 'Leales activos'      # compran seguido y recientemente
    elif r == 3 and f == 1:
        return 'Nuevos prometedores' # compraron hace poco pero solo una vez
    elif r == 2 and f >= 2:
        return 'En riesgo leve'      # compraban bien, se están enfriando
    elif r == 1 and f >= 2:
        return 'En riesgo alto'      # antes eran buenos clientes, llevan mucho sin comprar
    elif r == 1 and f == 1 and m == 1:
        return 'Perdidos'            # baja recencia, baja frecuencia, bajo gasto
    else:
        return 'Ocasionales'         # el resto

rfm['segmento'] = rfm.apply(segmentar, axis=1)

print(rfm['segmento'].value_counts())
print()
print(rfm.groupby('segmento')[['recency', 'frequency', 'monetary']].mean().round(1))


segmento
Ocasionales            166
En riesgo leve         161
Leales activos         126
Perdidos               101
En riesgo alto          97
Nuevos prometedores     77
Campeones               65
Name: count, dtype: int64

                     recency  frequency  monetary
segmento                                         
Campeones               22.6        9.8    5408.5
En riesgo alto         251.1        7.6    3613.1
En riesgo leve          79.2        7.8    3621.4
Leales activos          23.0        7.4    2660.5
Nuevos prometedores     20.7        4.2    1717.2
Ocasionales            202.2        4.2    2647.4
Perdidos               413.3        3.3     688.6


## Visualizar — scatter Frequency vs Monetary coloreado por segmento

In [5]:
fig = px.scatter(
    rfm,
    x='frequency',
    y='monetary',
    color='segmento',
    size='monetary',
    size_max=30,
    hover_data=['nombre', 'recency', 'rfm_score'],
    title='Segmentación RFM — Frequency vs Monetary',
    labels={
        'frequency': 'Frecuencia (nº de órdenes)',
        'monetary':  'Monetario (USD total)',
        'segmento':  'Segmento',
    },
    opacity=0.75,
)
fig.update_layout(legend_title='Segmento')
fig.show()


## Visualizar — distribución de segmentos por revenue total

In [6]:
resumen_seg = (
    rfm.groupby('segmento')
    .agg(
        clientes       = ('Customer ID', 'count'),
        revenue_total  = ('monetary',    'sum'),
        recency_media  = ('recency',     'mean'),
        freq_media     = ('frequency',   'mean'),
    )
    .reset_index()
    .sort_values('revenue_total', ascending=False)
    .round(1)
)

print(resumen_seg.to_string(index=False))

fig = px.bar(
    resumen_seg,
    x='segmento',
    y='revenue_total',
    color='segmento',
    text='clientes',
    title='Revenue total por segmento (número = clientes en ese grupo)',
    labels={'revenue_total': 'Revenue (USD)', 'segmento': ''},
)
fig.update_traces(texttemplate='%{text} clientes', textposition='outside')
fig.update_layout(showlegend=False, yaxis_tickformat='$,.0f')
fig.show()


           segmento  clientes  revenue_total  recency_media  freq_media
     En riesgo leve       161       583038.5           79.2         7.8
        Ocasionales       166       439470.9          202.2         4.2
          Campeones        65       351555.7           22.6         9.8
     En riesgo alto        97       350469.0          251.1         7.6
     Leales activos       126       335226.7           23.0         7.4
Nuevos prometedores        77       132227.1           20.7         4.2
           Perdidos       101        69548.8          413.3         3.3


## Interpretar los segmentos — qué hacer con cada uno

| Segmento | Acción sugerida |
|----------|-----------------|
| **Campeones** | Premiar, pedir reseñas, involucrar en lanzamientos nuevos |
| **Leales activos** | Programa de fidelización, venta cruzada |
| **Nuevos prometedores** | Onboarding: email de bienvenida, descuento en segunda compra |
| **En riesgo leve** | Reactivar: oferta personalizada antes de que se enfríen más |
| **En riesgo alto** | Campaña agresiva de recuperación: descuento importante |
| **Perdidos** | Bajo ROI — solo campañas de bajo costo (email genérico) |
| **Ocasionales** | Identificar qué los frena para convertirlos en leales |

La segmentación no sirve si no va acompañada de una acción diferente para cada grupo. Tratar igual a un campeón y a un cliente perdido es desperdiciar presupuesto de marketing.

---
# Bloque 2 — Diagnóstico de negocio

El rol del analista no termina en calcular métricas. El trabajo real es conectar los números con una causa raíz y una recomendación accionable.

La estructura del diagnóstico es siempre la misma:
1. Métricas generales — ¿cuál es el estado actual?
2. Comparación con período anterior — ¿sube o baja?
3. Descomposición por dimensión — ¿qué parte lo explica?
4. Causa raíz — ¿es un problema de volumen o de valor?
5. Recomendación — "X cayó un Y% porque Z, sugiero W"

## 1 — Métricas generales

In [7]:
# Separar por año para comparar
df['año'] = df['Order Date'].dt.year

metricas = (
    df.groupby('año')
    .agg(
        revenue         = ('Sales',       'sum'),
        clientes_unicos = ('Customer ID', 'nunique'),
        ordenes         = ('Order ID',    'nunique'),
    )
    .assign(
        ticket_medio = lambda x: (x['revenue'] / x['ordenes']).round(2),
        ordenes_x_cliente = lambda x: (x['ordenes'] / x['clientes_unicos']).round(2),
    )
    .reset_index()
)

print(metricas.to_string(index=False))


 año     revenue  clientes_unicos  ordenes  ticket_medio  ordenes_x_cliente
2015 479856.2081              589      947        506.71               1.61
2016 459436.0054              567     1019        450.87               1.80
2017 600192.5500              635     1295        463.47               2.04
2018 722052.0192              690     1661        434.71               2.41


## 2 — Variación interanual — ¿sube o baja?

In [8]:
# pct_change() calcula la variación respecto al período anterior
variacion = metricas.set_index('año')[['revenue', 'clientes_unicos', 'ordenes', 'ticket_medio']]
variacion_pct = variacion.pct_change().mul(100).round(1)

print('Variación % vs año anterior:')
print(variacion_pct.dropna().to_string())

# Visualizar revenue por año
fig = px.bar(
    metricas,
    x='año',
    y='revenue',
    text='revenue',
    title='Revenue total por año',
    labels={'revenue': 'Revenue (USD)', 'año': 'Año'},
    color_discrete_sequence=['steelblue'],
)
fig.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig.update_layout(yaxis_tickformat='$,.0f', showlegend=False)
fig.show()


Variación % vs año anterior:
      revenue  clientes_unicos  ordenes  ticket_medio
año                                                  
2016     -4.3             -3.7      7.6         -11.0
2017     30.6             12.0     27.1           2.8
2018     20.3              8.7     28.3          -6.2


## 3 — Descomposición por categoría — ¿qué impulsa el cambio?

In [9]:
rev_cat_año = (
    df.groupby(['año', 'Category'])['Sales']
    .sum()
    .reset_index()
    .rename(columns={'Sales': 'revenue'})
)

# Variación por categoría entre el último y penúltimo año
ultimo     = df['año'].max()
penultimo  = ultimo - 1

pivot_cat = (
    rev_cat_año[rev_cat_año['año'].isin([penultimo, ultimo])]
    .pivot(index='Category', columns='año', values='revenue')
    .assign(variacion_pct = lambda x: ((x[ultimo] - x[penultimo]) / x[penultimo] * 100).round(1))
    .assign(variacion_abs = lambda x: (x[ultimo] - x[penultimo]).round(0))
    .sort_values('variacion_pct', ascending=False)
)

print(f'Comparación {penultimo} vs {ultimo} por categoría:')
print(pivot_cat.to_string())

fig = px.bar(
    rev_cat_año,
    x='año',
    y='revenue',
    color='Category',
    barmode='group',
    title='Revenue por categoría y año',
    labels={'revenue': 'Revenue (USD)', 'año': 'Año'},
)
fig.update_layout(yaxis_tickformat='$,.0f')
fig.show()


Comparación 2017 vs 2018 por categoría:
año                    2017         2018  variacion_pct  variacion_abs
Category                                                              
Office Supplies  182417.566  240367.5410           31.8        57950.0
Technology       221961.944  269370.6910           21.4        47409.0
Furniture        195813.040  212313.7872            8.4        16501.0


## 4 — Descomposición por región — ¿dónde está el problema?

In [10]:
rev_reg_año = (
    df.groupby(['año', 'Region'])['Sales']
    .sum()
    .reset_index()
    .rename(columns={'Sales': 'revenue'})
)

pivot_reg = (
    rev_reg_año[rev_reg_año['año'].isin([penultimo, ultimo])]
    .pivot(index='Region', columns='año', values='revenue')
    .assign(variacion_pct = lambda x: ((x[ultimo] - x[penultimo]) / x[penultimo] * 100).round(1))
    .sort_values('variacion_pct', ascending=False)
)

print(f'Variación {penultimo} → {ultimo} por región:')
print(pivot_reg.to_string())

fig = px.line(
    rev_reg_año,
    x='año',
    y='revenue',
    color='Region',
    markers=True,
    title='Evolución de revenue por región',
    labels={'revenue': 'Revenue (USD)', 'año': 'Año'},
)
fig.update_layout(yaxis_tickformat='$,.0f')
fig.show()


Variación 2017 → 2018 por región:
año             2017         2018  variacion_pct
Region                                          
West     182471.2285  248130.9255           36.0
South     93535.9035  122164.5675           30.6
East     178511.5380  210129.1860           17.7
Central  145673.8800  141627.3402           -2.8


## 5 — Causa raíz — ¿menos clientes o menor gasto por cliente?

Cuando el revenue cae (o sube), hay dos causas posibles que llevan a acciones distintas:

- **Problema de adquisición**: hay menos clientes únicos → invertir en marketing/captación
- **Problema de monetización**: los mismos clientes gastan menos → revisar precios, mix de producto, retención

La descomposición `revenue = clientes × órdenes_por_cliente × ticket_medio` permite aislar cuál es el driver.

In [11]:
# Descomponer revenue en sus factores
diagnostico = metricas[metricas['año'].isin([penultimo, ultimo])].copy()

for col in ['revenue', 'clientes_unicos', 'ordenes_x_cliente', 'ticket_medio']:
    vals = diagnostico.set_index('año')[col]
    cambio = ((vals.iloc[-1] - vals.iloc[0]) / vals.iloc[0] * 100).round(1)
    print(f'{col:<25} {penultimo}: {vals.iloc[0]:>10,.1f}   {ultimo}: {vals.iloc[-1]:>10,.1f}   cambio: {cambio:+.1f}%')

print()
print('Conclusión:')
rev_cambio  = ((diagnostico.set_index('año').loc[ultimo, 'revenue'] -
                diagnostico.set_index('año').loc[penultimo, 'revenue']) /
               diagnostico.set_index('año').loc[penultimo, 'revenue'] * 100).round(1)
cli_cambio  = ((diagnostico.set_index('año').loc[ultimo, 'clientes_unicos'] -
                diagnostico.set_index('año').loc[penultimo, 'clientes_unicos']) /
               diagnostico.set_index('año').loc[penultimo, 'clientes_unicos'] * 100).round(1)
tick_cambio = ((diagnostico.set_index('año').loc[ultimo, 'ticket_medio'] -
                diagnostico.set_index('año').loc[penultimo, 'ticket_medio']) /
               diagnostico.set_index('año').loc[penultimo, 'ticket_medio'] * 100).round(1)

driver = 'adquisición (menos clientes)' if abs(cli_cambio) > abs(tick_cambio) else 'monetización (menor ticket medio)'
print(f'Revenue cambió {rev_cambio:+.1f}%.')
print(f'El driver principal es {driver}.')
print(f'  Clientes únicos: {cli_cambio:+.1f}%')
print(f'  Ticket medio:    {tick_cambio:+.1f}%')


revenue                   2017:  600,192.6   2018:  722,052.0   cambio: +20.3%
clientes_unicos           2017:      635.0   2018:      690.0   cambio: +8.7%
ordenes_x_cliente         2017:        2.0   2018:        2.4   cambio: +18.1%
ticket_medio              2017:      463.5   2018:      434.7   cambio: -6.2%

Conclusión:
Revenue cambió +20.3%.
El driver principal es adquisición (menos clientes).
  Clientes únicos: +8.7%
  Ticket medio:    -6.2%


## 6 — Recomendación final estructurada

In [12]:
# Construir el mensaje de diagnóstico de forma programática
# para que se actualice solo si cambian los datos

cat_mejor  = pivot_cat['variacion_pct'].idxmax()
cat_peor   = pivot_cat['variacion_pct'].idxmin()
reg_mejor  = pivot_reg['variacion_pct'].idxmax()
reg_peor   = pivot_reg['variacion_pct'].idxmin()

print('=' * 60)
print('DIAGNÓSTICO DE NEGOCIO — ESTRUCTURA FINAL')
print('=' * 60)
print(f"""
QUÉ PASÓ:
  Revenue {penultimo}→{ultimo}: {rev_cambio:+.1f}%

POR QUÉ:
  Driver principal: {driver}
  Clientes únicos cambiaron {cli_cambio:+.1f}%
  Ticket medio cambió {tick_cambio:+.1f}%

DÓNDE:
  Categoría que más creció: {cat_mejor} ({pivot_cat.loc[cat_mejor, 'variacion_pct']:+.1f}%)
  Categoría que más cayó:   {cat_peor}  ({pivot_cat.loc[cat_peor,  'variacion_pct']:+.1f}%)
  Región que más creció:    {reg_mejor} ({pivot_reg.loc[reg_mejor, 'variacion_pct']:+.1f}%)
  Región con peor evolución: {reg_peor} ({pivot_reg.loc[reg_peor,  'variacion_pct']:+.1f}%)

RECOMENDACIÓN:
  Si el driver es adquisición → revisar campañas de captación, 
  especialmente en {reg_peor} y en {cat_peor}.
  Si el driver es monetización → revisar pricing, mix de producto
  y estrategia de upsell en los segmentos RFM de bajo ticket.
""")


DIAGNÓSTICO DE NEGOCIO — ESTRUCTURA FINAL

QUÉ PASÓ:
  Revenue 2017→2018: +20.3%

POR QUÉ:
  Driver principal: adquisición (menos clientes)
  Clientes únicos cambiaron +8.7%
  Ticket medio cambió -6.2%

DÓNDE:
  Categoría que más creció: Office Supplies (+31.8%)
  Categoría que más cayó:   Furniture  (+8.4%)
  Región que más creció:    West (+36.0%)
  Región con peor evolución: Central (-2.8%)

RECOMENDACIÓN:
  Si el driver es adquisición → revisar campañas de captación, 
  especialmente en Central y en Furniture.
  Si el driver es monetización → revisar pricing, mix de producto
  y estrategia de upsell en los segmentos RFM de bajo ticket.



---
## Resumen del día

**RFM** → segmentar clientes por comportamiento para actuar diferente con cada grupo.

**Diagnóstico** → la estructura `qué pasó → por qué → dónde → qué hacer` convierte los números en decisiones accionables.

| Concepto | Técnica pandas |
|----------|----------------|
| Recency | `groupby + lambda: (FECHA_REF - x.max()).days` |
| Frequency | `nunique('Order ID')` |
| Monetary | `sum('Sales')` |
| Scoring | `pd.qcut(col, q=3, labels=[1,2,3])` |
| Variación % | `pct_change().mul(100)` |
| Descomposición | `pivot(index, columns, values)` |
| Causa raíz | comparar `clientes_unicos` vs `ticket_medio` |
